# R08-H54 - The Bayesian layer is numerology at 44% calibration

**Author**: Knowledge Graph Foundry autonomous build (kj) <br>
**Date**: 2026-07-07 <br>
**Pipeline stage**: R08 contrarian round, deterministic replay <br>
**Graph**: rebuilt CPAP graph (neo4j2, read-only) <br>

Registered test: decision replay over the LOGGED cross-type decisions shows >=90% agreement with a
two-rule (name-identity + embedding) system; among disagreements the audit sides with the
deterministic rule at least as often as the posterior. Refuted if the posterior's defer zone
uniquely prevents measured false merges the two rules would commit.

Variant that ran: **logged decisions exist** (logs/kgf-events.jsonl carries resolution.merge/defer/block
with full posterior components) - this is the registered primary path, not the DEFERRED reconstruction fallback.

In [1]:
# Imports
# stdlib
import json, datetime, collections
# third party
import numpy as np
from neo4j import GraphDatabase
from rich import print as rprint

NEO4J_URI = 'bolt://user-konrad.jelen-kgf-neo4j2:7687'  # read-only neo4j2 (NOT .env, which points at the live graph)
EVENTS = '../logs/kgf-events.jsonl'
T_MERGE = 0.60   # posterior >= 0.60 -> merge (empirical boundary)
T_BLOCK = 0.40   # posterior <  0.40 -> block; between -> defer
Z = 1.96
rprint(f'[bold]config[/bold] thresholds merge>={T_MERGE} block<{T_BLOCK}')

config thresholds merge>=0.6 block<0.4

## Load logged decisions

In [2]:
rows=[]
for line in open(EVENTS):
    line=line.strip()
    if not line: continue
    e=json.loads(line)
    if e.get('event') in ('resolution.merge','resolution.defer','resolution.block'):
        rows.append(e)
N=len(rows)
dec_counts=collections.Counter(r['decision'] for r in rows)
rprint(f'[bold]{N}[/bold] logged resolution decisions:', dict(dec_counts))

# formula verification: posterior_odds = prior_odds * lr_desc * lr_emb * lr_cooc
def post_from(prior, ld, le, lc):
    o=(prior/(1-prior))*ld*le*lc
    return o/(1+o)
errs=[abs(post_from(r['prior'],r['lr_description'],r['lr_embedding'],r['lr_cooccurrence'])-r['posterior']) for r in rows]
rprint('formula max reconstruction error:', round(max(errs),6))

4625 logged resolution decisions:
{'defer': 843, 'block': 468, 'merge': 3314}

formula max reconstruction error: 0.0

## LR distributions - which signals actually vary

In [3]:
for f in ['prior','lr_embedding','lr_description','lr_cooccurrence','posterior']:
    v=np.array([r[f] for r in rows])
    rprint(f'{f:16s} uniq={len(set(np.round(v,4))):5d} min={v.min():.3f} med={np.median(v):.3f} max={v.max():.3f}')
rprint('lr_cooccurrence values:', collections.Counter(round(r['lr_cooccurrence'],3) for r in rows).most_common())
# how often does each extra-machinery LR sit at its neutral/modal value
n_desc_neutral=sum(1 for r in rows if abs(r['lr_description']-1.0)<1e-9)
rprint(f'lr_description exactly 1.0 (neutral): {n_desc_neutral}/{N} = {n_desc_neutral/N:.1%}')

prior            uniq=  667 min=0.200 med=0.621 max=0.790

lr_embedding     uniq= 3110 min=0.299 med=1.741 max=1.997

lr_description   uniq=  110 min=0.300 med=1.200 max=2.000

lr_cooccurrence  uniq=    2 min=0.900 med=1.500 max=1.500

posterior        uniq= 3123 min=0.100 med=0.770 max=0.956

lr_cooccurrence values:
[(1.5, 3989), (0.9, 636)]

lr_description exactly 1.0 (neutral): 286/4625 = 6.2%

## Two-rule counterfactual - drop description + cooccurrence LRs

The two-rule system is name-identity (the prior) plus embedding (lr_embedding). The description and
cooccurrence LRs are the extra Bayesian machinery H54 calls decoration. Neutralise them (set to 1.0),
recompute the posterior from prior x lr_embedding only, apply the SAME thresholds, and measure 3-way
decision agreement. If agreement >=90% the extra machinery does not change decisions.

In [4]:
def decide(p):
    return 'merge' if p>=T_MERGE else ('block' if p<T_BLOCK else 'defer')

# actual 3-way decision from logged posterior (reconstruct with thresholds to confirm labels match)
actual=[decide(r['posterior']) for r in rows]
agree_actual=sum(a==r['decision'] for a,r in zip(actual,rows))/N
rprint(f'threshold labels match logged decision: {agree_actual:.1%}')

# neutral counterfactual: desc=cooc=1.0
cf=[decide(post_from(r['prior'],1.0,r['lr_embedding'],1.0)) for r in rows]
agree_neutral=sum(a==b for a,b in zip(cf,actual))/N
rprint(f'[bold]two-rule (name+embedding) agreement with posterior: {agree_neutral:.1%}[/bold]')

# confusion of disagreements
dis=[(a,b) for a,b in zip(actual,cf) if a!=b]
conf=collections.Counter(dis)
rprint('disagreement transitions posterior->tworule:', {f'{a}->{b}':n for (a,b),n in conf.most_common()})

# also: cooccurrence-only drop and description-only drop, to attribute the gap
cf_desc=[decide(post_from(r['prior'],1.0,r['lr_embedding'],r['lr_cooccurrence'])) for r in rows]
cf_cooc=[decide(post_from(r['prior'],r['lr_description'],r['lr_embedding'],1.0)) for r in rows]
rprint(f'drop description only -> agreement {sum(a==b for a,b in zip(cf_desc,actual))/N:.1%}')
rprint(f'drop cooccurrence only -> agreement {sum(a==b for a,b in zip(cf_cooc,actual))/N:.1%}')

threshold labels match logged decision: 100.0%

two-rule (name+embedding) agreement with posterior: 74.4%

disagreement transitions posterior->tworule:
{
    'defer->merge': 400,
    'merge->defer': 321,
    'block->defer': 214,
    'defer->block': 194,
    'block->merge': 42,
    'merge->block': 15
}

drop description only -> agreement 78.2%

drop cooccurrence only -> agreement 83.4%

## Best-case two-rule - recalibrated thresholds on (prior, lr_embedding)

Give the two-rule system its best shot: recompute posterior' from prior x lr_embedding (desc=cooc=1.0),
then sweep new cut points (t_lo', t_hi') to maximise 3-way agreement with the actual decisions. If even
the best-calibrated two-feature system cannot reach 90%, the extra LRs are load-bearing.

In [5]:
pcf=np.array([post_from(r['prior'],1.0,r['lr_embedding'],1.0) for r in rows])
actual_arr=np.array(actual)
best=0; best_t=None
grid=np.linspace(0.2,0.9,71)
for tl in grid:
    for th in grid:
        if th<=tl: continue
        pred=np.where(pcf>=th,'merge',np.where(pcf<tl,'block','defer'))
        ag=(pred==actual_arr).mean()
        if ag>best: best=ag; best_t=(round(tl,3),round(th,3))
rprint(f'[bold]best-case recalibrated two-rule agreement: {best:.1%}[/bold] at (t_lo,t_hi)={best_t}')

best-case recalibrated two-rule agreement: 76.6% at (t_lo,t_hi)=(0.31, 0.42)

## Audit adjudication - disagreements against the labeled sets

Map the disagreeing entity-id pairs to names and cross-reference the labeled sets (66-pair inventory
causes, 6 labeled false merges, 252-pair frozen labels). The defer-zone clause: does the posterior's
defer/block uniquely prevent a labeled false merge that the two-rule would merge?

In [6]:
drv=GraphDatabase.driver(NEO4J_URI, auth=('neo4j','kgfoundry'), notifications_min_severity='OFF')
with drv.session() as s:
    idname={r['id']:r['name'] for r in s.run('MATCH (e:Entity) RETURN e.id AS id, e.name AS name').data()}
drv.close()

# labeled pairs (name-based) -> label
import glob
lab={}
mr=json.load(open(sorted(glob.glob('../reports/matching-r12-foundation-*.json'))[-1]))
for p in mr['pairs']:
    lab[frozenset((p['a'].lower(),p['b'].lower()))]=p['y']  # y=1 same, 0 different
ff=json.load(open(sorted(glob.glob('../reports/identity-forensics-r11-final-*.json'))[-1]))
false_pairs=set(frozenset((x['a'].lower(),x['b'].lower())) for x in ff['labeled_false_merges'])

# for each decision, attach names + any label
def names(r):
    return idname.get(r['left_id'],'?'), idname.get(r['right_id'],'?')
labeled_decisions=[]
for r,a,c in zip(rows,actual,cf):
    na,nb=names(r)
    key=frozenset((na.lower(),nb.lower()))
    y=lab.get(key)
    isfalse=key in false_pairs
    if y is not None or isfalse:
        labeled_decisions.append(dict(a=na,b=nb,posterior=round(r['posterior'],3),post_dec=a,tworule=c,y=y,labeled_false=isfalse))
rprint(f'logged decisions that hit a labeled pair: {len(labeled_decisions)}')
for d in labeled_decisions[:40]:
    rprint(d)

logged decisions that hit a labeled pair: 33

{
    'a': 'C-Flex',
    'b': 'C-Flex+',
    'posterior': 0.611,
    'post_dec': 'merge',
    'tworule': 'merge',
    'y': 1,
    'labeled_false': False
}

{
    'a': 'Smart Ramp',
    'b': 'SmartRamp',
    'posterior': 0.624,
    'post_dec': 'merge',
    'tworule': 'merge',
    'y': 1,
    'labeled_false': False
}

{
    'a': 'AirSense 10 AutoSet',
    'b': 'AirSense 10 AutoSet for Her',
    'posterior': 0.514,
    'post_dec': 'defer',
    'tworule': 'merge',
    'y': 1,
    'labeled_false': False
}

{
    'a': 'AirFit F20 for Her',
    'b': 'AirFit F10 for Her',
    'posterior': 0.711,
    'post_dec': 'merge',
    'tworule': 'merge',
    'y': 0,
    'labeled_false': False
}

...

{
    'a': 'Double-sided tape for 934 sensor',
    'b': 'Double sided tape for 935 and 953',
    'posterior': 0.661,
    'post_dec': 'merge',
    'tworule': 'merge',
    'y': 0,
    'labeled_false': False
}

{
    'a': 'Amara full face mask',
    'b': 'Amara',
    'posterior': 0.294,
    'post_dec': 'block',
    'tworule': 'block',
    'y': 1,
    'labeled_false': False
}

{
    'a': 'Host software manual v2.0 (International English)',
    'b': 'Host software manual v2.0 (French)',
    'posterior': 0.775,
    'post_dec': 'merge',
    'tworule': 'merge',
    'y': 1,
    'labeled_false': False
}

{
    'a': 'Host software manual v2.0 (International English)',
    'b': 'Host software manual v2.0 (Spanish)',
    'posterior': 0.789,
    'post_dec': 'merge',
    'tworule': 'merge',
    'y': 1,
    'labeled_false': False
}

{
    'a': 'Host software manual v2.0 (French)',
    'b': 'Host software manual v2.0 (Spanish)',
    'posterior': 0.863,
    'post_dec': 'merge',
    'tworule': 'merge',
    'y': 1,
    'labeled_false': False
}

In [7]:
# Defer-zone clause: labeled-false pairs the posterior did NOT merge but the two-rule WOULD merge
defer_saves=[d for d in labeled_decisions if d['labeled_false'] and d['post_dec']!='merge' and d['tworule']=='merge']
# and: labeled-false pairs the posterior merged (a real false merge committed by the resolver)
post_false_merges=[d for d in labeled_decisions if d['labeled_false'] and d['post_dec']=='merge']
# among y-labeled disagreements, who is right
adj=[d for d in labeled_decisions if d['y'] is not None and d['post_dec']!=d['tworule']]
def correct(dec,y):  # merge correct iff y==1
    if dec=='merge': return y==1
    return y==0  # defer/block treated as 'not merge'
post_right=sum(correct(d['post_dec'],d['y']) for d in adj)
two_right=sum(correct(d['tworule'],d['y']) for d in adj)
rprint(f'defer-zone unique saves of labeled false merges: {len(defer_saves)}')
rprint(f'labeled false merges the posterior committed: {len(post_false_merges)}')
rprint(f'y-labeled disagreements: {len(adj)} | posterior correct {post_right} vs two-rule correct {two_right}')

defer-zone unique saves of labeled false merges: 0

labeled false merges the posterior committed: 0

y-labeled disagreements: 5 | posterior correct 1 vs two-rule correct 4

## Verdict + report

In [8]:
confirm = agree_neutral>=0.90 and len(defer_saves)==0
verdict = 'CONFIRMED' if confirm else 'REFUTED'
# refutation is specifically triggered if defer zone uniquely prevents a labeled false merge
if len(defer_saves)>0:
    verdict='REFUTED'; reason='defer zone uniquely prevents labeled false merge the two-rule would commit'
elif agree_neutral>=0.90:
    reason='two-rule reproduces >=90% of decisions; extra LRs decoration'
else:
    reason=f'two-rule reproduces only {agree_neutral:.1%} (<90%); description LR is load-bearing'
    verdict='REFUTED'

report=dict(
  hypothesis='R08-H54', variant='logged-decisions',
  n_decisions=N, decision_counts=dict(dec_counts),
  lr_cooccurrence_values=dict(collections.Counter(round(r['lr_cooccurrence'],3) for r in rows)),
  lr_description_uniq=len(set(round(r['lr_description'],4) for r in rows)),
  lr_description_neutral_frac=n_desc_neutral/N,
  two_rule_agreement=agree_neutral,
  drop_description_only_agreement=sum(a==b for a,b in zip(cf_desc,actual))/N,
  drop_cooccurrence_only_agreement=sum(a==b for a,b in zip(cf_cooc,actual))/N,
  best_case_recalibrated_agreement=best, best_thresholds=best_t,
  labeled_decisions=len(labeled_decisions),
  defer_zone_unique_false_merge_saves=len(defer_saves),
  defer_saves=defer_saves,
  posterior_committed_false_merges=len(post_false_merges),
  y_labeled_disagreements=len(adj), posterior_correct=post_right, two_rule_correct=two_right,
  bar='>=90% agreement + audit parity; refuted if defer zone uniquely prevents a labeled false merge',
  verdict=verdict, reason=reason)
stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d-%H%M%S')
path=f'../reports/bayesian-replay-h54-{stamp}.json'
json.dump(report, open(path,'w'), indent=2)
rprint(f'[bold green]{verdict}[/bold green] - {reason}')
rprint('wrote', path)

REFUTED - two-rule reproduces only 74.4% (<90%); description LR is load-bearing

wrote ../reports/bayesian-replay-h54-20260707-092554.json